In [ ]:
%pip install riskfolio-lib
%pip install dwave

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time
import logging
import os
import warnings
import itertools
from datetime import datetime, timedelta
from functools import partial
from pathlib import Path

SEED = 12
np.random.seed(SEED)
msg_level = logging.INFO
# Suppress all RuntimeWarnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Portfolio Optimization using multiple engines

In [ ]:
# Create a logger
logger = logging.getLogger("inspect_results_logger")
logger.setLevel(msg_level)  # Set the level for this logger

# Create a handler (where to send the logs)
handler = logging.StreamHandler()  # Send to the console
handler.setLevel(msg_level)

# Create a formatter (how to format the logs)
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

# Add the handler to the logger
logger.addHandler(handler)

## Path Definition

1. Create a folder called Datasets in Drive
2. Copy the files from Github to this folder

In [ ]:
benchmark_path = '/content/drive/MyDrive/Datasets/benchmark_gspc.pkl'
source_path = '/content/drive/MyDrive/Datasets/stocks_adjclose.pkl'

## Load benchmark

In [ ]:
benchmark = pd.read_pickle(benchmark_path)
sns.lineplot(benchmark['^GSPC'])
benchmark.head()

## Load Source

In [ ]:
source = pd.read_pickle(source_path)
print(source.shape)
# Check if any value in the DataFrame is null
has_any_nan = source.isnull().values.any()
print("Any NaN in source_wide:", has_any_nan)
source.head()

## Correlation Analysis

In [ ]:
df_corr = source.corr()
df_corr.head()


## Rank correlation ascending

In [ ]:
# rank by correlation
corr_sum = df_corr.map(lambda x: abs(x)).sum()
corr_rank = corr_sum.sort_values().rank(method='min').astype(int)
corr_rank

## Rank returns descending

In [ ]:
# rank by returns
return_rank = source.diff().sum(axis=0).sort_values().rank(method='min', ascending=False).astype(int)
return_rank

## Select sets of 10 stocks

In [ ]:
select_10 = (return_rank + corr_rank).sort_values().reset_index()['Ticker'].values[:11]
select_10

## Portfolio Stats

In [ ]:
def portfolio_stats(weights, data):

    weights = np.array(weights)
    returns =np.log(data) - np.log(data.shift(1)) # log return to minimize fp error
    #returns = data.pct_change().dropna()
    port_return = np.sum(returns.mean() * weights)
    port_vol = np.sqrt(np.dot(weights.T, np.dot(returns.cov() , weights)))
    try:
        sharpe_ratio = port_return/port_vol
    except Exception as e:
        sharpe_ratio = 0
    return sharpe_ratio, port_return, port_vol

## Fitness function

In [ ]:
def fitness_function(weights, data):
    sharpe_ratio, _, _ = portfolio_stats(weights, data)
    return sharpe_ratio

## Minimize Sharpe

In [ ]:
def curried_fitness(weights, data):
    port_return = partial(fitness_function, data)
    return port_return

def minimize_sharpe(weights):
    return -1 * curried_fitness(weights)

## Data Generator

In [ ]:
def generate_data(df, benchmark, days_to_avg=30, days_to_opt=30, seed=SEED):
    df2 = df.reset_index()
    benchmark2 = benchmark.reset_index()
    elements = df2.sample(n=100, random_state=seed).index # definint a maximum of 100 different sampled initial dates
    for idx in elements:
        df_sample = df2.iloc[idx-days_to_avg:idx+days_to_opt, :]
        df_sample = df_sample.set_index('ds')
        df_sample_b = benchmark2.iloc[idx-days_to_avg:idx+days_to_opt, :]
        df_sample_b = df_sample_b.set_index('ds').drop(['index'], axis=1)
        yield df_sample, df_sample_b


## Backtest

In [ ]:
def backtest(optimization_function, data, benchmark, initial_capital, avg_period, opt_period):
    portfolio_value = initial_capital
    portfolio_returns = []
    benchmark_returns = []
    portfolio_total_return = []
    portfolio_sharpe_ratios = []
    weights_history = pd.DataFrame(index=data.index, columns=data.columns)
    portfolio_value_history = pd.Series(index=data.index, name='Portfolio Value', dtype='float')
    portfolio_value_history.iloc[0] = portfolio_value


    j = 0
    for i in range(avg_period+1, avg_period + opt_period+1):
        df = data.iloc[j:i, :]
        df_pct = df.iloc[j:i, :].pct_change().dropna(axis=0)
        #logger.debug(f'df_pct: {df_pct}')
        weights = optimization_function(df)
        weights[weights < 0] = 0
        weights /= weights.sum()
        weights_history.loc[df.index[-1]] = weights
        #sharpe_ratio, portfolio_return, portfolio_volatility = portfolio_stats(weights, df)
        # portfolio_change = df.iloc[-2:, :].pct_change() * weights
        portfolio_change = df_pct.iloc[-1] * weights
        #portfolio_return = portfolio_change.sum(axis=1).iloc[-1]
        portfolio_return = portfolio_change.sum()
        portfolio_returns.append(portfolio_return)
        # print(f'portfolio returns: {portfolio_returns}')
        benchmark_return = benchmark.iloc[j:i, :].pct_change().iloc[-1].values.tolist()[0]
        benchmark_returns.append(benchmark_return)
        # print(f'benchmark_returns: {benchmark_returns}')
        portfolio_cumulative_returns = np.cumprod([k + 1 for k in portfolio_returns])
        # print(f'portfolio cumulative returns: {portfolio_cumulative_returns}')
        benchmark_cumulative_returns = np.cumprod([k + 1 for k in  benchmark_returns])
        # print(f'benchmark cumulative returns: {benchmark_cumulative_returns}')
        portfolio_mean_return = np.mean(portfolio_returns)
        benchmark_mean_return = np.mean(benchmark_returns)
        portfolio_volatility = np.std(portfolio_returns)
        benchmark_volatility = np.std(benchmark_returns)
        try:
            sharpe_ratio = (portfolio_mean_return) / portfolio_volatility
        except Exception as e:
            sharpe_ratio = 0
        portfolio_sharpe_ratios.append(sharpe_ratio)

         # Portfolio & Benchmark value
        benchmark_value = initial_capital * benchmark_cumulative_returns[-1]
        portfolio_value = initial_capital * portfolio_cumulative_returns[-1]
        j += 1

    portfolio_cumulative_returns = portfolio_cumulative_returns - portfolio_cumulative_returns[0]
    benchmark_cumulative_returns = benchmark_cumulative_returns - benchmark_cumulative_returns[0]

    # Plot the results
    plt.figure(figsize=(12, 6))
    plt.plot(portfolio_cumulative_returns, label='Portfolio')
    plt.plot(benchmark_cumulative_returns, label='Benchmark')
    #plt.plot(portfolio_returns, label='Portfolio')
    #plt.plot(benchmark_returns, label='Benchmark')
    plt.legend(loc='upper left')
    plt.title('Backtesting Results')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Returns')
    plt.show()

    return weights_history, portfolio_value_history, portfolio_cumulative_returns, benchmark_cumulative_returns

## Read-write pickle

In [ ]:
import pickle

def write_pickle_dict(data, file_path):
    """Pickles a dictionary and saves it to a file."""
    try:
        with open(file_path, 'wb') as f:  # Open the file in binary write mode ('wb')
            pickle.dump(data, f)
        print(f"Dictionary pickled and saved to {file_path}")
    except Exception as e:
        print(f"An error occurred while pickling: {e}")

def read_pickle_dict(file_path):
    try:
        with open(file_path, 'rb') as f:
            loaded_dict = pickle.load(f)
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")
    return loaded_dict



## Run Experiment

In [ ]:
def run_experiment(results_path_template, data, benchmark, opt_fun, parameters):

    n_periods = parameters['n_periods']
    days_to_avg = parameters['days_to_avg']
    days_to_opt = parameters['days_to_opt']
    population_size = parameters['population_size']
    num_generations = parameters['num_generations']
    initial_capital = parameters['initial_capital']
    datagen = generate_data(data, benchmark)


    for i in range(n_periods):
        results_path = results_path_template.format(i)
        if not os.path.exists(results_path):
            try:
                df, df_b = next(datagen)
                print(f'initial date: {df.iloc[days_to_avg+1:, :].index[0]}')
                plt.figure(figsize=(12, 6))
                plt.plot(df.iloc[days_to_avg+1:days_to_avg+days_to_opt+1, :].sum(axis=1), label='Source')
                plt.plot(df_b, label='Benchmark')
                plt.legend(loc='upper left')
                plt.title('Source Data')
                plt.xlabel('Date')
                plt.ylabel('Stock Values')
                plt.show()
                start_time = datetime.now()
                weights_history, portfolio_value_history, portfolio_cumulative_returns, benchmark_cumulative_returns = backtest(opt_fun, df, df_b, initial_capital=initial_capital, avg_period=days_to_avg,opt_period=days_to_opt)
                logger.debug(f'portfolio cumulative returns: {portfolio_cumulative_returns}')
                end_time = datetime.now()
                dt = abs(end_time - start_time)
            except Exception as e:
                print(f'Failed {results_path} due to {e}')
                weights_history, portfolio_value_history, portfolio_cumulative_returns, benchmark_cumulative_returns = [], [], [], []

            result = {
                "round": i,
                "start_date": df.index[0],
                "end_date": df.index[-1],
                "population_size": population_size,
                "num_generations": num_generations,
                "days_to_avg": days_to_avg,
                "days_to_opt": days_to_opt,
                "weights_history": weights_history,
                "portfolio_value_history": portfolio_value_history,
                "portfolio_cumulative_returns": portfolio_cumulative_returns,
                "benchmark_cumulative_returns": benchmark_cumulative_returns,
                "total_run_time": dt.total_seconds()
                }
            write_pickle_dict(result, results_path)

        else:
            results = read_pickle_dict(results_path)
            portfolio_cumulative_returns = results['portfolio_cumulative_returns']
            benchmark_cumulative_returns = results['benchmark_cumulative_returns']

            # Plot the results
            plt.figure(figsize=(12, 6))
            plt.plot(portfolio_cumulative_returns, label='Portfolio')
            plt.plot(benchmark_cumulative_returns, label='Benchmark')
            plt.legend(loc='upper left')
            plt.title('Backtesting Results')
            plt.xlabel('Date')
            plt.ylabel('Cumulative Returns')
            plt.show()

    return None

In [ ]:
def trio_comparison(results_path_1_template,
                    results_path_2_template,
                    results_path_3_template,
                    n_periods):


    for i in range(n_periods):
        results_path_1 = results_path_1_template.format(i)
        results_path_2 = results_path_2_template.format(i)
        results_path_3 = results_path_3_template.format(i)
        try:
            results_1 = read_pickle_dict(results_path_1)
        except Exception as e:
            logger.error(f'Error reading {results_path_1}: {e}')
        try:
            results_2 = read_pickle_dict(results_path_2)
        except Exception as e:
            logger.error(f'Error reading {results_path_2}: {e}')
        try:
            results_3 = read_pickle_dict(results_path_3)
        except Exception as e:
            logger.error(f'Error reading {results_path_3}: {e}')

        if results_1 is not None:
            portfolio_cumulative_returns_1 = results_1['portfolio_cumulative_returns']
            benchmark_cumulative_returns_1 = results_1['benchmark_cumulative_returns']
            initial_date_1 = results_1['start_date']
        if results_2 is not None:
            portfolio_cumulative_returns_2 = results_2['portfolio_cumulative_returns']
            benchmark_cumulative_returns_2 = results_2['benchmark_cumulative_returns']
            initial_date_2 = results_2['start_date']
        if results_3 is not None:
            portfolio_cumulative_returns_3 = results_3['portfolio_cumulative_returns']
            benchmark_cumulative_returns_3 = results_3['benchmark_cumulative_returns']
            initial_date_3 = results_3['start_date']

        # Plot the results
        logger.info(f'Initial date 1: {initial_date_1}')
        logger.info(f'Initial date 2: {initial_date_2}')
        logger.info(f'Initial date 3: {initial_date_3}')
        plt.figure(figsize=(12, 6))
        if results_1 is not None:
            plt.plot(portfolio_cumulative_returns_1, label=f'{results_path_1}')
        if results_2 is not None:
            plt.plot(portfolio_cumulative_returns_2, label=f'{results_path_2}')
        if results_3 is not None:
            plt.plot(portfolio_cumulative_returns_3, label=f'{results_path_3}')
        if results_1 is not None:
            plt.plot(benchmark_cumulative_returns_1, label='Benchmark 1')
        if results_2 is not None:
            plt.plot(benchmark_cumulative_returns_2, label='Benchmark 2')
        if results_3 is not None:
            plt.plot(benchmark_cumulative_returns_3, label='Benchmark 3')
        plt.legend(loc='upper left')
        plt.title('Backtesting Results')
        plt.xlabel('Date')
        plt.ylabel('Cumulative Returns')
        plt.show()

## Optimization Engines

In [ ]:
import numpy as np
import scipy.optimize as optimize
import riskfolio as rp
import logging
from functools import partial
seed = 12
np.random.seed(seed)
logger = logging.getLogger("inspect_results_logger")

# Definition of the Genetic Algorithm with elitism. I am only considering solutions where all the weights are > 0. That might not be optimal for comparison with GSPC but it is the best mapping to Capital Allocation.

def genetic_algorithm(data, fitness_function, population_size=500, num_generations=1000, mutation_rate=0.05, elitism=0.1):
    population = np.random.rand(population_size, len(data.columns))
    population = population / np.sum(population, axis=1)[:, np.newaxis]
    fitness = np.array([fitness_function(individual, data) for individual in population])
    for generation in range(num_generations):
        sorted_idx = np.argsort(fitness)[::-1]
        population = population[sorted_idx]
        fitness = fitness[sorted_idx]
        num_elites = int(elitism * population_size)
        offspring = population[:num_elites]
        parent1_idx = np.random.randint(num_elites, population_size, size=population_size-num_elites)
        parent2_idx = np.random.randint(num_elites, population_size, size=population_size-num_elites)
        parent1 = population[parent1_idx]
        parent2 = population[parent2_idx]
        crossover_prob = np.random.rand(population_size-num_elites, len(data.columns))
        crossover_mask = crossover_prob <= 0.5
        offspring_crossover = np.where(crossover_mask, parent1, parent2)
        mutation_prob = np.random.rand(population_size-num_elites, len(data.columns))
        mutation_mask = mutation_prob <= 0.5
        mutation_values = np.random.rand(population_size-num_elites, len(data.columns))
        mutation_direction = np.random.choice([-1, 1], size=(population_size - num_elites, len(data.columns)))
        offspring_mutation = np.where(mutation_mask, offspring_crossover + mutation_direction * mutation_values, offspring_crossover)
        population = np.vstack((population[:num_elites], offspring_mutation))
        fitness = np.array([fitness_function(individual, data) for individual in population])
    selected = []
    # consider only solutions where all weights are greater than zero
    #logger.debug(f'fitness: {fitness}')
    for f in fitness:
        if np.all(f > 0):
            selected.append(f)
    best_idx = np.argmax(selected)
    best_individual = population[best_idx]
    logger.debug('### Best Individual ###')
    logger.debug(best_individual)

    return best_individual


def scipy_minimize(data, fitness_function):
    num_assets = data.shape[1]
    constraints = ({'type' : 'eq', 'fun': lambda x: np.sum(x) -1})
    bounds = tuple((0.01, 0.2) for x in range(num_assets))
    initializer = num_assets * [1./num_assets,]
    port_return = partial(fitness_function, data=data)

    def minimize_sharpe(weights):
        return -1 * port_return(weights)

    weights = np.random.dirichlet(np.ones(num_assets),size=1)
    optimal_sharpe=optimize.minimize(minimize_sharpe,
                                    initializer,
                                    method = 'SLSQP',
                                    bounds = bounds,
                                    constraints = constraints)

    optimal_sharpe_weights=optimal_sharpe['x'].round(4)
    return np.array(optimal_sharpe_weights)


def riskfolio_minimize(data):
    #y = np.log(data) - np.log(data.shift(1))
    y = np.log(data) - np.log(data.shift(1))
    port = rp.HCPortfolio(returns=y[1:])

    # Estimate optimal portfolio:

    model='HERC' # Could be HRP or HERC
    codependence = 'pearson' # Correlation matrix used to group assets in clusters
    rm = 'MV' # Risk measure used, this time will be variance
    rf = 0 # Risk free rate
    linkage = 'single' # Linkage method used to build clusters
    max_k = 10 # Max number of clusters used in two difference gap statistic, only for HERC model
    leaf_order = True # Consider optimal order of leafs in dendrogram

    w = port.optimization(model=model,
                        codependence=codependence,
                        rm=rm,
                        rf=rf,
                        linkage=linkage,
                        max_k=max_k,
                        leaf_order=leaf_order)

    return np.array(w).flatten()

In [ ]:
parameters = {
    "population_size": 100,
    "num_generations": 100,
    "mutation_rate": 0.1,
    "elitism": 0.1,
    "n_periods": 10,
    "days_to_avg": 30,
    "days_to_opt": 30,
    "initial_capital": 1000,
}

In [ ]:
data_10 = source[select_10]
sns.set_style('darkgrid')
data_10.plot(figsize=(12,6))
plt.legend(loc='upper left')
plt.show()

In [ ]:
df = data_10.copy()
days_to_avg = parameters['days_to_avg']
display(df.head())
print(f'initial date: {df.iloc[days_to_avg+1, :].index}')

In [ ]:
opt_fun_ga = partial(genetic_algorithm,
                  fitness_function=fitness_function,
                  population_size=parameters["population_size"],
                  num_generations=parameters["num_generations"],
                  mutation_rate=parameters["mutation_rate"],
                  elitism=parameters["elitism"])

In [ ]:
results_path_ga_10 = '../results/results_10_{}_ga.pkl'
_ = run_experiment(results_path_ga_10, data_10, benchmark, opt_fun_ga, parameters)

In [ ]:
opt_fun_bf = partial(scipy_minimize, fitness_function=fitness_function)
results_path_bf_10 = '../results/results_10_{}_bf.pkl'
_ = run_experiment(results_path_bf_10, data_10, benchmark, opt_fun_bf, parameters)

In [ ]:
opt_fun_hrp = partial(riskfolio_minimize)
results_path = '../results/results_10_{}_hrp.pkl'
_ = run_experiment(results_path, data_10, benchmark, opt_fun_hrp, parameters)

## d-wave Exact

In [ ]:
def dwave_exact(data):

    solver = ExactCQMSolver()
    cqm = ConstrainedQuadraticModel()
    budget = 1000
    alpha = 0.5

    # Get prices and calculate max shares
    price = data.iloc[-1,:]  # Use latest prices
    max_num_shares = (budget/price).astype(int)
    stocks = data.columns.tolist()

    # Calculate returns and covariance
    log_returns = np.log(data) - np.log(data.shift(1))
    avg_daily_returns = log_returns.mean()
    covariance_matrix = log_returns.cov()  # Use log returns for covariance

    # Define integer variables for each stock
    x = {s: Integer(f"{s}", lower_bound=0, upper_bound=max_num_shares[s]) for s in stocks}

    # Calculate risk term (using log returns covariance)
    risk = 0
    for s1, s2 in product(stocks, stocks):
        coeff = covariance_matrix.loc[s1, s2] * price[s1] * price[s2]
        risk = risk + coeff * x[s1] * x[s2]

    # Calculate returns term (using log returns)
    returns = 0
    for s in stocks:
        returns = returns + price[s] * avg_daily_returns[s] * x[s]

    # Add constraints and objective
    cqm.add_constraint(quicksum([x[s]*price[s] for s in stocks]) <= budget, label='upper_budget')
    cqm.set_objective(alpha * risk - returns)

    try:
        # Solve the model
        #sample_set = solver.sample_cqm(cqm)
        sample_set = solver.sample(cqm)
        feasible_samples = sample_set.filter(lambda d: d.is_feasible)

        if not feasible_samples:
            logger.warning("No feasible solution found, returning equal weights")
            return np.ones(len(stocks)) / len(stocks)

        # Get best solution
        best_feasible = feasible_samples.first
        solution = {k: int(best_feasible.sample[k]) for k in stocks}

        # Convert to weights
        weights = np.array([solution[s] * price[s] for s in stocks])
        weights = weights / np.sum(weights) if np.sum(weights) > 0 else np.ones(len(stocks)) / len(stocks)

        return weights

    except Exception as e:
        logger.error(f"Error in D-Wave solver: {str(e)}")
        return np.ones(len(stocks)) / len(stocks)

